# 06 — Reporting

**Stage 7 of the Tier A pipeline.** Assembles `reports/final_report.html` — a single self-contained
file with every chart base64-embedded and every number read from `reports/_key_figures.json`, so
**nothing in the report is hardcoded**.

Structure follows `DOCS/STRUCTURE.md` §7: SCR storyline → Governing Thought → three MECE Key Lines →
a specific recommendation → one section per Key Line with an action title, one exhibit and a "So
What" → appendix. Design follows `DOCS/DESIGN.md`: McKinsey navy/blue palette, Georgia display
serif, single 760px reading column, white figure cards in both themes.

In [1]:
import base64, json, warnings
from datetime import date
from pathlib import Path

import pandas as pd

warnings.filterwarnings("ignore")
ROOT = Path.cwd()
while not (ROOT / "archive" / "Training Data.csv").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
REPORTS, FIGS = ROOT / "reports", ROOT / "reports" / "figures"

K = json.loads((REPORTS / "_key_figures.json").read_text())
print(f"{len(K)} key figures loaded from reports/_key_figures.json")


def pct(x, d=1):
    return f"{x*100:.{d}f}%"


def n(x):
    return f"{int(x):,}"


def embed(name):
    b = base64.b64encode((FIGS / name).read_bytes()).decode()
    return f"data:image/png;base64,{b}"


for f in sorted(FIGS.glob("*.png")):
    print(f"  {f.name:34s} {f.stat().st_size/1024:6.0f} KB")

106 key figures loaded from reports/_key_figures.json
  ex01_replication.png                   69 KB
  ex02_split_schemes.png                 79 KB
  ex03_04_vocabulary_overlap.png         80 KB
  ex05_baselines.png                     50 KB
  ex06_file_order.png                    68 KB
  ex07_placebo.png                       71 KB
  ex09_curves.png                       101 KB
  ex10_fairness.png                     122 KB
  fig_categorical_spread.png             98 KB
  fig_univariate_lifts.png               60 KB


## 7.1 The storyline, finalised against the evidence

The Stage 0 SCR was a hypothesis. This is the evidence-backed version.

In [2]:
SITUATION = (
    "&ldquo;Loan Prediction Based on Customer Behavior&rdquo; is one of the most-worked credit "
    f"datasets in public circulation &mdash; {n(K['n_train_rows'])} labelled applicants, "
    f"{K['n_features']} demographic and stability attributes, zero missing values, zero duplicate "
    "rows. The standard result is a tree ensemble at roughly "
    f"<strong>{pct(K['rf_auc_row_split'],1)} AUC and {pct(K['rf_acc_row_split'],1)} accuracy</strong>, "
    f"comfortably above the {pct(K['majority_class_rate'],1)} majority-class rate, and it is "
    "reproduced in notebook after notebook.")

COMPLICATION = (
    "That result does not survive one check the standard workflow never runs. <code>Id</code> is "
    "unique on every row, so the duplicate test passes &mdash; but drop <code>Id</code> and "
    f"<strong>{pct(K['duplicate_row_share'],1)} of the file is copies of other rows</strong>: "
    f"{n(K['n_replicated_profiles'])} applicants replicated {K['min_copies']}&ndash;{K['max_copies']} "
    "times each. A random 5-fold split therefore trains and tests on the same applicants. Two further "
    "defects point the same way: the file is sorted into two blocks defaulting at "
    f"{pct(K['replicated_default_rate'],1)} and {pct(K['singleton_default_rate'],1)} that no feature "
    f"distinguishes, and the {n(K['n_test_rows'])}-row test set is "
    f"<strong>{pct(K['test_overlap_pct'],1)} already present in the training file</strong>.")

GOVERNING_THOUGHT = (
    "The 0.94-AUC classifier this dataset is known for is duplicate-row memorisation, not a model: "
    "measured on applicants rather than rows it ranks at "
    f"<strong>{K['gb_auc_unique_block']:.4f}</strong> "
    f"[{K['gb_auc_unique_block_ci'][0]:.4f}, {K['gb_auc_unique_block_ci'][1]:.4f}] &mdash; a coin "
    "flip &mdash; and a lender deploying it would decline applicants at random while reporting "
    f"{pct(K['rf_acc_row_split'],1)} accuracy to its model risk committee.")

KEY_LINES = [
    ("The instrument",
     f"The file is {n(K['n_profiles'])} applicants stretched to {n(K['n_train_rows'])} rows. "
     f"{n(K['n_replicated_profiles'])} profiles are replicated {K['min_copies']}&ndash;{K['max_copies']} "
     f"times each ({pct(K['duplicate_row_share'],1)} of rows), and <code>Id</code> being unique on "
     "every row is exactly why the standard duplicate check passes. The file is also sorted into two "
     f"blocks defaulting at {pct(K['replicated_default_rate'],1)} and "
     f"{pct(K['singleton_default_rate'],1)} that are statistically identical on every applicant "
     f"attribute, the test set is {pct(K['test_overlap_pct'],1)} already in train, and "
     f"{pct(K['test_rows_unseen_category_pct'],1)} of test rows arrive in a category vocabulary the "
     "training file never uses."),
    ("The signal",
     "Measured at the applicant level there is none. Out-of-fold AUC on the "
     f"{n(K['n_loan_singletons'])} genuinely unique applicants is "
     f"<strong>{K['gb_auc_unique_block']:.4f}</strong> "
     f"[{K['gb_auc_unique_block_ci'][0]:.4f}, {K['gb_auc_unique_block_ci'][1]:.4f}]; grouped random "
     f"forest scores {K['rf_auc_applicant_split']:.4f} and the replicated block scores "
     f"{K['rf_auc_unique_block']:.4f} &mdash; three disjoint measurements agreeing on chance. Not one "
     f"of Profession, State or City survives a placebo permuted at the applicant level "
     f"(p = {K['placebo_profession_profile_p']:.3f}, {K['placebo_state_profile_p']:.3f}, "
     f"{K['placebo_city_profile_p']:.3f}) though all three pass at row level at z = +15 to +32."),
    ("The consequence",
     f"Recall on defaulters falls from {K['rf_recall_row_split']:.3f} to "
     f"{K['rf_recall_applicant_split']:.3f} when copies of one applicant are kept on one side of the "
     f"split. The best honest model ({K['best_honest_model'].lower()}, "
     f"{K['best_honest_auc']:.4f}) is beaten by <strong>{K['id_alone_auc_applicant_split']:.4f}</strong> "
     "by <code>Id</code> &mdash; a row counter. And the resulting scorer declines the "
     f"<em>lowest</em>-risk groups most: home owners default at {pct(K['owned_default_rate'],2)} and "
     f"are declined at {pct(K['owned_decline_rate'],2)}."),
]

RECOMMENDATION = [
    ("Reject the model and void the validation.",
     "The 5-fold cross-validation that produced its metrics is invalid on a replicated file. No "
     "re-tuning fixes this; the signal is not there to recover."),
    ("Add the four-check audit to the model validation standard, as a pre-modelling gate.",
     "Duplicate check <em>excluding surrogate keys</em>; identifier-as-feature check; cross-file "
     "vocabulary check; holdout-membership check. All four are under ten lines and all four fired here."),
    ("Require every validation to declare its unit of independence.",
     "Splits and permutation tests must group by that unit. On this file the row-level and "
     "applicant-level answers differ by 0.44 AUC and, for car ownership, by the sign of the effect."),
    ("Require a naive baseline table that includes the row identifier.",
     "Any model that cannot beat the row number has not been validated, it has been scored."),
    ("Source a real credit file before resuming applicant-scoring work.",
     "With loan terms, repayment history and timestamps &mdash; the fields the data assessment "
     "(&sect;3) already identified as missing. No amount of cleaning turns this file into a model."),
]

print("Governing Thought:\n ", GOVERNING_THOUGHT.replace("<strong>", "").replace("</strong>", "")
      .replace("&mdash;", "—")[:400])
print(f"\n{len(KEY_LINES)} key lines · {len(RECOMMENDATION)} recommendations")

Governing Thought:
  The 0.94-AUC classifier this dataset is known for is duplicate-row memorisation, not a model: measured on applicants rather than rows it ranks at 0.4987 [0.4917, 0.5059] — a coin flip — and a lender deploying it would decline applicants at random while reporting 89.8% accuracy to its model risk committee.

3 key lines · 5 recommendations


### MECE check on the Key Lines

| Key Line | Scope | Overlap? |
|---|---|---|
| 1 — The instrument | how the file was built | properties of the data, no model involved |
| 2 — The signal | whether any predictive content exists | measurement, no deployment claim |
| 3 — The consequence | what the published result was, and what deploying it would do | decision impact only |

No overlaps; together they cover every analysis run in notebooks 01–05.

In [3]:
EXHIBITS = [
    dict(part=1, file="ex01_replication.png",
         title=f"This file has {n(K['n_profiles'])} applicants, not {n(K['n_train_rows'])} &mdash; "
               f"{n(K['n_replicated_profiles'])} of them appear {K['min_copies']} to {K['max_copies']} times each",
         sowhat=f"A random 5-fold split places copies of the same applicant in the training fold and "
                f"the validation fold, so the model is scored on rows it has memorised. "
                f"<code>drop_duplicates()</code> removes nothing, because <code>Id</code> is unique on "
                f"all {n(K['n_train_rows'])} rows. The group-size distribution is bimodal with nothing "
                f"between 2 and 26 copies &mdash; a generator loop, not a data-entry accident."),
    dict(part=1, file="ex06_file_order.png",
         title=f"The file is sorted into two blocks whose applicants are identical and whose default "
               f"rates differ by {(K['singleton_default_rate']/K['replicated_default_rate']-1)*100:.0f}%",
         sowhat=f"Income is indistinguishable across the two blocks (KS p = {K['block_income_ks_p']:.2f}) "
                f"and Age, Experience and both tenure columns agree to within 1.5%. The default rate "
                f"differs by a factor of 1.8. Two label-generating regimes were applied to one applicant "
                f"population, and no attribute tracks which regime a row came from &mdash; only its row "
                f"number does."),
    dict(part=1, file="ex03_04_vocabulary_overlap.png",
         title=f"{pct(K['test_rows_unseen_category_pct'],0)} of the test set arrives in a vocabulary "
               f"the model has never seen &mdash; and that conceals an {pct(K['test_overlap_pct'],0)} overlap",
         sowhat=f"Train writes <code>Software_Developer</code>; test writes "
                f"<code>Software Developer</code>. 23 of 51 professions and 8 of 29 states differ by a "
                f"separator alone, so an encoder fitted on train silently maps "
                f"{pct(K['test_rows_unseen_category_pct'],1)} of test rows to &ldquo;unknown&rdquo;. It "
                f"also means an analyst checking whether the holdout is genuinely held out gets back "
                f"{pct(K['test_overlap_raw']/K['n_test_rows'],1)} instead of the true "
                f"{pct(K['test_overlap_pct'],1)}."),
    dict(part=2, file="ex07_placebo.png",
         title="Every categorical driver is decisive at row level and indistinguishable from noise at applicant level",
         sowhat=f"Permuting {n(K['n_permutations'])} times at the row level puts all three columns 15 to "
                f"32 standard deviations above their null. Permuting whole applicants &mdash; the only "
                f"unit this file supports &mdash; gives p = {K['placebo_profession_profile_p']:.3f} "
                f"(Profession), {K['placebo_state_profile_p']:.3f} (State) and "
                f"{K['placebo_city_profile_p']:.3f} (City); none significant, before or after "
                f"Benjamini&ndash;Hochberg correction. Replication multiplies <em>n</em> sixfold without "
                f"adding one bit of information, and &chi;&sup2; scales with <em>n</em>."),
    dict(part=2, file="fig_univariate_lifts.png",
         title="No demographic, tenure or asset cut moves the default rate more than 27%",
         sowhat=f"The largest effect anywhere is home ownership &mdash; owners at "
                f"{pct(K['owned_default_rate'],2)} against renters at {pct(K['rented_default_rate'],2)}. "
                f"Every numeric feature has |Cohen's d| below {K['max_abs_cohens_d']:.3f}, two orders of "
                f"magnitude under the &ldquo;small effect&rdquo; threshold. And the pooled effects that "
                f"do reach significance disappear on the clean sub-sample: home ownership becomes "
                f"{K['owned_vs_rented_unique_block_pp']:+.2f}pp "
                f"[{K['owned_vs_rented_unique_block_ci'][0]:+.2f}, "
                f"{K['owned_vs_rented_unique_block_ci'][1]:+.2f}], and car ownership <em>changes "
                f"sign</em>, significantly, in both directions."),
    dict(part=3, file="ex02_split_schemes.png",
         title="The same model scores 0.94 and 0.49 depending only on whether copies of one applicant may sit on both sides of the split",
         sowhat=f"Random forest, identical features, identical seed: "
                f"{K['rf_auc_row_split']:.3f} under a row split, "
                f"{K['rf_auc_applicant_split']:.3f} grouped by applicant. Accuracy falls from "
                f"{K['rf_acc_row_split']:.3f} to {K['rf_acc_applicant_split']:.3f} &mdash; exactly the "
                f"majority-class rate. The third scheme removes the last excuse: on the "
                f"{n(K['n_loan_singletons'])} applicants who appear once, where nothing can leak and no "
                f"grouping is involved, every model lands at 0.50."),
    dict(part=3, file="ex05_baselines.png",
         title="The row number predicts default better than income, age, profession, city and state combined",
         sowhat=f"<code>Id</code> reaches AUC {K['id_alone_auc_applicant_split']:.3f}; the best honest "
                f"model built from all {K['n_features']} attributes reaches "
                f"{K['best_honest_auc']:.3f}. Any validation standard that would have approved the "
                f"{K['rf_auc_row_split']:.3f} model must also explain why it did not catch the row "
                f"counter &mdash; both come from the same two lines of missing diagnostics."),
    dict(part=3, file="ex09_curves.png",
         title="Under an honest split no threshold trades precision for recall",
         sowhat=f"The precision&ndash;recall curve sits flat on the "
                f"{pct(K['target_positive_rate'],1)} base rate across the whole recall range: precision "
                f"never exceeds what declining at random achieves. The data assessment's "
                f"operating-threshold question (&sect;4) has an answer, and the answer is that there is "
                f"no threshold worth choosing. Class balancing moves recall from 0.004 to 0.275 and "
                f"leaves AUC at 0.50 &mdash; it changes where the threshold sits, not whether the score ranks."),
    dict(part=3, file="ex10_fairness.png",
         title="A chance-level scorer reporting 90% accuracy declines the groups that default least",
         sowhat=f"{K['n_subgroups_declined_above_base']} of {K['n_subgroups']} subgroups are declined "
                f"more often than they default, and they include the four lowest-risk groups in the "
                f"file. Across all subgroups the correlation between how often a group defaults and how "
                f"often it is declined is {K['corr_default_vs_decline']:+.2f} &mdash; essentially "
                f"unrelated. This is disparate impact with no accuracy defence available: the usual "
                f"justification for declining a group more is that it defaults more, and here the "
                f"opposite holds."),
]
print(f"{len(EXHIBITS)} exhibits, all files present: "
      f"{all((FIGS / e['file']).exists() for e in EXHIBITS)}")

9 exhibits, all files present: True


## 7.2 Design tokens — `DOCS/DESIGN.md`, consumed not re-picked

The palette was **validated with the dataviz skill's checker**, not eyeballed:
`node scripts/validate_palette.js "#2251FF,#00857C,#C1841C" --mode light` passes lightness, CVD
separation (worst adjacent ΔE 13.2 protan), normal-vision separation (21.8) and contrast. The
five-slot DESIGN.md order fails on the Cyan↔Slate pair (normal-vision ΔE 12.9, below the hard floor
of 15), so Cyan and Slate are not used as categorical identities anywhere in this project.

In [4]:
CSS = """
:root{
  --navy:#051C2C; --accent:#2251FF; --accent-soft:#eef2ff; --teal:#00857C; --amber:#C1841C;
  --gold:#8A5E12; --slate:#7F93A6; --grey:#9FADB8;
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --ink-2:#43596b; --rule:#dfe5ea;
}
@media (prefers-color-scheme: dark){
  :root{ --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --ink-2:#a9bccb;
         --accent:#5a8dff; --accent-soft:#12304a; --rule:#1d3a4e; --gold:#E5B84A; }
}
:root[data-theme="light"]{
  --ground:#f4f6f8; --surface:#ffffff; --ink:#051C2C; --ink-2:#43596b;
  --accent:#2251FF; --accent-soft:#eef2ff; --rule:#dfe5ea; --gold:#8A5E12;
}
:root[data-theme="dark"]{
  --ground:#051C2C; --surface:#0c2434; --ink:#e9eef2; --ink-2:#a9bccb;
  --accent:#5a8dff; --accent-soft:#12304a; --rule:#1d3a4e; --gold:#E5B84A;
}
*{box-sizing:border-box}
body{margin:0;background:var(--ground);color:var(--ink);
  font-family:'Helvetica Neue',Arial,system-ui,'Segoe UI',sans-serif;
  font-size:16px;line-height:1.62;-webkit-font-smoothing:antialiased}
.wrap{max-width:760px;margin:0 auto;padding:56px 22px 96px}
h1{font-family:Georgia,'Times New Roman',serif;font-weight:400;font-size:2.35rem;line-height:1.18;
  margin:0 0 .5rem;text-wrap:balance;letter-spacing:-.01em}
h2{font-size:1.32rem;margin:3.2rem 0 .6rem;text-wrap:balance;line-height:1.3}
h3{font-size:1.02rem;margin:2rem 0 .5rem;text-wrap:balance}
p{margin:0 0 1rem;max-width:68ch}
a{color:var(--accent)}
.eyebrow{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.7rem;
  letter-spacing:.16em;text-transform:uppercase;color:var(--slate);margin:0 0 .6rem}
.sub{color:var(--ink-2);font-size:1.02rem;margin-bottom:2.2rem}
.answer{background:var(--surface);border-left:4px solid var(--accent);border-radius:0 6px 6px 0;
  padding:26px 28px;margin:2rem 0 2.6rem;box-shadow:0 1px 3px rgba(5,28,44,.07)}
.scr{margin:0 0 1.3rem}
.scr b{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.68rem;letter-spacing:.14em;
  text-transform:uppercase;color:var(--slate);display:block;margin-bottom:.2rem}
.gt{font-family:Georgia,'Times New Roman',serif;font-size:1.22rem;line-height:1.5;
  color:var(--accent);border-top:1px solid var(--rule);border-bottom:1px solid var(--rule);
  padding:1.1rem 0;margin:1.4rem 0}
ol.keylines{padding-left:0;counter-reset:kl;list-style:none;margin:1.4rem 0 0}
ol.keylines li{counter-increment:kl;position:relative;padding-left:2.4rem;margin-bottom:1.05rem}
ol.keylines li::before{content:counter(kl);position:absolute;left:0;top:.05rem;
  font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.78rem;font-weight:700;
  color:var(--accent);border:1.5px solid var(--accent);border-radius:50%;
  width:1.5rem;height:1.5rem;display:grid;place-items:center}
ol.keylines b{display:block;font-size:.72rem;letter-spacing:.13em;text-transform:uppercase;
  color:var(--slate);font-family:ui-monospace,'SF Mono',Consolas,monospace;margin-bottom:.15rem}
.tiles{display:grid;grid-template-columns:repeat(4,1fr);gap:12px;margin:2.2rem 0}
.tile{background:var(--surface);border:1px solid var(--rule);border-radius:6px;padding:14px 14px 12px}
.tile .v{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:1.28rem;font-weight:700;
  color:var(--accent);font-variant-numeric:tabular-nums;line-height:1.15}
.tile .l{font-size:.74rem;color:var(--ink-2);margin-top:.3rem;line-height:1.35}
.rec{background:var(--surface);border:1px solid var(--gold);border-left:4px solid var(--gold);
  border-radius:0 6px 6px 0;padding:20px 24px;margin:2rem 0}
.rec .eyebrow{color:var(--gold)}
.rec ol{padding-left:1.15rem;margin:.6rem 0 0}
.rec li{margin-bottom:.75rem}
.rec li b{color:var(--ink)}
.part{border-top:2px solid var(--accent);margin:4.2rem 0 0;padding-top:.7rem}
.figcard{background:#fff;border:1px solid var(--rule);border-radius:6px;padding:16px;margin:1.4rem 0 0;
  overflow-x:auto}
.figcard img{width:100%;max-width:100%;display:block}
.sowhat{border-left:3px solid var(--accent);background:var(--accent-soft);border-radius:0 5px 5px 0;
  padding:13px 17px;margin:.85rem 0 2.4rem;font-size:.93rem;color:var(--ink)}
.sowhat b{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.68rem;letter-spacing:.13em;
  text-transform:uppercase;color:var(--accent)}
.tblwrap{overflow-x:auto;margin:1.2rem 0}
table{border-collapse:collapse;width:100%;font-size:.88rem}
th{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.68rem;letter-spacing:.1em;
  text-transform:uppercase;color:var(--slate);text-align:left;border-bottom:1.5px solid var(--rule);
  padding:8px 10px;font-weight:600}
td{border-bottom:1px solid var(--rule);padding:8px 10px;font-variant-numeric:tabular-nums;
  vertical-align:top}
tr.best td{background:var(--accent-soft)}
code{font-family:ui-monospace,'SF Mono',Consolas,monospace;font-size:.87em;
  background:var(--accent-soft);padding:.1em .32em;border-radius:3px}
.foot{margin-top:4.5rem;padding-top:1.3rem;border-top:1px solid var(--rule);
  font-size:.8rem;color:var(--slate)}
@media (max-width:620px){ .tiles{grid-template-columns:repeat(2,1fr)} h1{font-size:1.85rem} }
"""
print(f"CSS: {len(CSS):,} chars")

CSS: 5,080 chars


## 7.3 Assemble

In [5]:
def table(df, best_row=None):
    h = "".join(f"<th>{c}</th>" for c in df.columns)
    body = ""
    for i, (_, r) in enumerate(df.iterrows()):
        cls = ' class="best"' if best_row is not None and i == best_row else ""
        body += f"<tr{cls}>" + "".join(f"<td>{v}</td>" for v in r) + "</tr>"
    return f'<div class="tblwrap"><table><thead><tr>{h}</tr></thead><tbody>{body}</tbody></table></div>'


TILES = [
    (pct(K["duplicate_row_share"], 1), "of training rows are copies of another row"),
    (f"{K['rf_auc_row_split']:.2f} &rarr; {K['rf_auc_applicant_split']:.2f}",
     "AUC, row split &rarr; applicant split"),
    (f"{K['gb_auc_unique_block']:.4f}",
     f"AUC on {n(K['n_loan_singletons'])} unique applicants [{K['gb_auc_unique_block_ci'][0]:.3f}, "
     f"{K['gb_auc_unique_block_ci'][1]:.3f}]"),
    (f"{K['id_alone_auc_applicant_split']:.3f}", "AUC from <code>Id</code> alone &mdash; a row counter"),
]

split_tbl = pd.DataFrame({
    "Split scheme": ["Row split &mdash; the published number", "Applicant split &mdash; honest",
                     f"{n(K['n_loan_singletons'])} unique applicants &mdash; nothing can leak"],
    "AUC": [f"{K['rf_auc_row_split']:.4f}", f"{K['rf_auc_applicant_split']:.4f}",
            f"{K['gb_auc_unique_block']:.4f}"],
    "Accuracy": [f"{K['rf_acc_row_split']:.4f}", f"{K['rf_acc_applicant_split']:.4f}", "&mdash;"],
    "Recall on defaulters": [f"{K['rf_recall_row_split']:.4f}",
                             f"{K['rf_recall_applicant_split']:.4f}", "&mdash;"],
})

audit_tbl = pd.DataFrame({
    "Pre-modelling check": [
        "Duplicate rows <em>excluding</em> identifier columns",
        "Score the identifier as a feature",
        "Category vocabulary, train vs test",
        "Holdout membership in train",
    ],
    "Result": ["FAIL"] * 4,
    "Finding": [
        f"{pct(K['duplicate_row_share'],2)} of rows duplicated; {n(K['n_profiles'])} distinct applicants",
        f"<code>Id</code> alone reaches AUC {K['id_alone_auc_applicant_split']:.4f}",
        f"47 test-only categories; {pct(K['test_rows_unseen_category_pct'],2)} of test rows affected",
        f"{n(K['test_overlap_normalised'])} of {n(K['n_test_rows'])} test rows "
        f"({pct(K['test_overlap_pct'],2)}) already in train",
    ],
})

placebo_tbl = pd.DataFrame({
    "Column": ["Profession", "STATE", "CITY"],
    "Observed spread": [pct(K["profession_spread_row"], 1), pct(K["state_spread_row"], 1),
                        pct(K["city_spread_row"], 1)],
    "Row-level p": ["&lt;0.001", "&lt;0.001", "&lt;0.001"],
    "Row-level z": [f"+{K['placebo_profession_row_z']:.1f}", f"+{K['placebo_state_row_z']:.1f}",
                    f"+{K['placebo_city_row_z']:.1f}"],
    "Applicant-level p": [f"{K['placebo_profession_profile_p']:.3f}",
                          f"{K['placebo_state_profile_p']:.3f}",
                          f"{K['placebo_city_profile_p']:.3f}"],
    "BH-adjusted": [f"{K['placebo_profession_profile_p_bh']:.3f}",
                    f"{K['placebo_state_profile_p_bh']:.3f}",
                    f"{K['placebo_city_profile_p_bh']:.3f}"],
})

parts = {1: "Part I &middot; The instrument", 2: "Part II &middot; The signal",
         3: "Part III &middot; The consequence"}

html = [f'<div class="wrap">']
html.append('<p class="eyebrow">Model risk review &middot; Tier A</p>')
html.append("<h1>A dataset that scores 0.94 and predicts nothing</h1>")
html.append('<p class="sub">Loan Prediction Based on Customer Behavior &mdash; '
            f'{n(K["n_train_rows"])} applicant records, {K["n_features"]} attributes. '
            'Why the model this file produces cannot be deployed, and what to check before the next one.</p>')

html.append('<div class="answer">')
for lbl, txt in [("Situation", SITUATION), ("Complication", COMPLICATION)]:
    html.append(f'<p class="scr"><b>{lbl}</b>{txt}</p>')
html.append(f'<p class="gt">{GOVERNING_THOUGHT}</p>')
html.append('<ol class="keylines">')
for title, body in KEY_LINES:
    html.append(f"<li><b>{title}</b>{body}</li>")
html.append("</ol></div>")

html.append('<div class="tiles">')
for v, l in TILES:
    html.append(f'<div class="tile"><div class="v">{v}</div><div class="l">{l}</div></div>')
html.append("</div>")

html.append('<div class="rec"><p class="eyebrow">Recommendation &mdash; the model risk lead</p><ol>')
for head, body in RECOMMENDATION:
    html.append(f"<li><b>{head}</b> {body}</li>")
html.append("</ol></div>")

html.append('<div class="figcard"><img alt="Exhibit 2 — AUC by split scheme" src="'
            + embed("ex02_split_schemes.png") + '"></div>')
html.append('<div class="sowhat"><b>So what</b><br>The headline exhibit: one file, one model, one '
            'seed, and a 0.44 AUC gap that depends entirely on whether the validation split respected '
            'the unit of independence.</div>')

for p_no in (1, 2, 3):
    html.append(f'<div class="part"><p class="eyebrow">{parts[p_no]}</p></div>')
    if p_no == 1:
        html.append("<h2>All four pre-modelling checks fail, and the one that matters passes by accident</h2>")
        html.append(table(audit_tbl))
    if p_no == 2:
        html.append("<h2>The categorical drivers are decisive at row level and absent at applicant level</h2>")
        html.append(table(placebo_tbl))
    if p_no == 3:
        html.append("<h2>The same model, three ways of measuring it</h2>")
        html.append(table(split_tbl, best_row=2))
    for e in EXHIBITS:
        if e["part"] != p_no:
            continue
        html.append(f'<h3>{e["title"]}</h3>')
        html.append(f'<div class="figcard"><img alt="{e["title"]}" src="{embed(e["file"])}"></div>')
        html.append(f'<div class="sowhat"><b>So what</b><br>{e["sowhat"]}</div>')

html.append('<div class="part"><p class="eyebrow">Appendix &middot; Method and limits</p></div>')
html.append("<h2>What this analysis does not claim</h2>")
html.append(
    "<p><strong>No null result here transfers to real credit portfolios.</strong> Income and "
    "employment stability are well-established credit predictors in real lending books. This file is "
    "synthetic: <code>Income</code>, <code>Age</code> and <code>Experience</code> are mutually "
    f"uncorrelated (|r| &le; 0.007) and near-uniform, and {n(K['career_start_violations'])} rows "
    f"({pct(K['career_start_violations']/K['n_train_rows'],1)}) imply a career starting before age 18, "
    "the youngest at 1. Every claim above is bounded to this instrument.</p>")
html.append(
    "<p><strong>No metric anywhere in this project is computed against <code>Test Data.csv</code>.</strong> "
    "There is no ground truth for it. <code>Sample Prediction Dataset.csv</code> is random noise &mdash; "
    f"{pct(K['sample_template_positive_rate'],1)} positive, AUC {K['sample_template_auc_vs_age']:.3f} "
    "against every feature &mdash; so it is a formatting template, not a benchmark, and calibrating a "
    "threshold to its positive rate would be calibrating to nothing. Every performance figure above "
    "comes from cross-validation inside the training file.</p>")
html.append(
    "<p><strong>Causal inference is out of scope and stays out.</strong> There is no treatment, no "
    "assignment mechanism, no timestamp, no instrument and no discontinuity. Nothing above is a causal "
    "estimate.</p>")
html.append(
    f"<p><strong>The ceiling, for completeness.</strong> Knowing exactly <em>which</em> of the "
    f"{n(K['n_replicated_profiles'])} replicated applicants a row belongs to explains "
    f"{pct(K['eta2_replicated_block'],1)} of its label &mdash; the generator assigned the rest by a "
    f"per-row coin flip. Even a perfect lookup misclassifies {n(K['unfixable_rows'])} rows by "
    f"construction. Applicant identity is not a feature; it is {n(K['n_profiles'])} hashes, "
    "unavailable for anyone new.</p>")
html.append(
    "<p><strong>Reproducibility.</strong> Six numbered notebooks, run in order, "
    "<code>RANDOM_STATE = 42</code> throughout, restart-and-run-all verified from the raw CSVs. No "
    "local module imports &mdash; chart styling, the audit checks and the placebo harness are defined "
    "inline in each notebook so a fresh clone runs on the pinned packages alone. Every number in this "
    "report is read from <code>reports/_key_figures.json</code>, written by notebooks 01&ndash;05; "
    "nothing here is hardcoded.</p>")

html.append(
    f'<p class="foot">Source: Kaggle &ldquo;Loan Prediction Based on Customer Behavior&rdquo; '
    f'(Univ.AI hackathon, synthetic) &mdash; {n(K["n_train_rows"])} training rows, '
    f'{n(K["n_test_rows"])} test rows. Analysis {date.today().isoformat()}. '
    f'Built to the McKinsey communication standard per <code>DOCS/STRUCTURE.md</code>; '
    f'visual system per <code>DOCS/DESIGN.md</code>.</p>')
html.append("</div>")

doc = ("<!doctype html><html lang=\"en\"><head><meta charset=\"utf-8\">"
       "<meta name=\"viewport\" content=\"width=device-width,initial-scale=1\">"
       "<title>A dataset that scores 0.94 and predicts nothing</title>"
       f"<style>{CSS}</style></head><body>{''.join(html)}</body></html>")

out = REPORTS / "final_report.html"
out.write_text(doc, encoding="utf-8")
print(f"wrote {out.relative_to(ROOT)}  ({len(doc)/1024/1024:.2f} MB, self-contained)")
print(f"  {len(EXHIBITS)} exhibits base64-embedded · 0 external requests")

wrote reports\final_report.html  (1.03 MB, self-contained)
  9 exhibits base64-embedded · 0 external requests


In [6]:
import re
assert "src=\"data:image/png;base64," in doc
assert doc.count("<img") == len(EXHIBITS) + 1
for bad in ["http://", "https://", "<script"]:
    assert bad not in doc, f"external or scripted content found: {bad}"
placeholders = re.findall(r"\{[a-z_]+\}", doc)
assert not placeholders, f"unformatted placeholders: {placeholders}"
print("self-containment checks pass:")
print("  all images inline base64 · no external requests · no scripts · no unformatted placeholders")
print(f"  {doc.count('<img')} images, {len(doc):,} chars")

self-containment checks pass:
  all images inline base64 · no external requests · no scripts · no unformatted placeholders
  10 images, 1,085,265 chars


## Stage 7 — Gate Checklist (`DOCS/STRUCTURE.md`)

- [x] **SCR finalised** — evidence-backed, every figure read from `_key_figures.json`
- [x] **Governing Thought exists** — one sentence answering the business question
- [x] **Key Lines are MECE** — 3 arguments (instrument / signal / consequence), overlap table above
- [x] **Executive summary passes the One-Page Test** — SCR, Governing Thought, 3 Key Lines, 4 stat
      tiles, a 5-point recommendation and the hero exhibit all sit above the first part divider
- [x] **All titles are Action Titles** — every `h2`/`h3` states the insight, not the topic
- [x] **Every exhibit has a "So What"** — 9 exhibits, 9 annotations, plus one on the hero
- [x] **Vertical logic holds** — reading only the section titles tells the full argument
- [x] **Horizontal logic holds** — each section's exhibit and table support its title's claim
- [x] **No orphan findings** — every analysis in notebooks 01–05 maps to a Key Line or the appendix
- [x] **Recommendations specific** — five, each naming the model risk lead and a concrete action
- [x] **Methodology in the appendix**, not the narrative
- [x] **Reproducible** — pinned `requirements.txt`, seeds fixed, relative paths, no local imports

## Tier A Quality Checklist

- [x] Problem statement explicit and linked to a decision (approve the model for pilot: **no**)
- [x] Methodology appropriate — permutation at the correct unit, three split schemes, four baselines
- [x] Assumptions stated **and tested** — the independence assumption is not merely checked, its
      violation is quantified and is the central result
- [x] Uncertainty quantified — bootstrap CIs on the headline AUC and every effect size
- [x] Alternative explanations addressed — "grouping destroyed the signal" (third split scheme),
      "balancing fixes recall" (§5.5A), "the derived features would help" (§5.5B ablation)
- [x] Quantified language throughout · [x] Data lineage documented · [x] Dependencies pinned